# 06-O3 -- Does the PM2.5 search choose ozone if it is offered? (Bogota)

`06_feature_selection.ipynb` removed ozone from the PM2.5 candidate pool as "not an independent driver". That is a
modelling choice, not a fact: ozone and its radicals do take part in forming secondary PM2.5. Rather than only
justifying the exclusion, this notebook **offers ozone and lets the search decide**.

Everything else is identical to notebook 06's same-day, no-target-lag run: same candidate pool (calendar terms and `pm25_lag1`
removed, >=70% coverage), same neural-network leave-one-year-out sequential forward selection
(`common.feature_selection.sequential_forward_selection`). The only change is that the paper's own ozone (MDA8, network mean,
from the ozone master dataset) and its lags 1-3 are added to the pool. To keep the comparison fair, both runs use the **same rows**
(days with an ozone value) and the lead PM2.5 feature set is re-scored on those rows.

Interpretation caveat, stated in advance: selecting ozone would show predictive information, not causation, and ozone and
PM2.5 share meteorology (and ozone's own equation contains PM2.5), so any use of ozone in the PM2.5 equation makes the two
equations circular. This is a sensitivity test, not a proposal to replace Eq. 1.

In [1]:
# %% Settings (edit here, then Run All)
MIN_COVERAGE = 0.70                 # as in notebook 06
MAX_FEATURES, MIN_GAIN = 12, 0.002  # as in common.feature_selection defaults
N_JOBS = 10
OZONE_LAGS = (1, 2, 3)              # lags of ozone offered together with same-day ozone
RUN_CONTROL = True                  # also re-run the selection WITHOUT ozone on the same rows

In [2]:
import sys, json, time
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd
from common.feature_selection import loyo_r2, sequential_forward_selection
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
df = pd.read_parquet(DATA_DIR / "bogota_master_dataset.parquet")
df = df.dropna(subset=["pm25"]).reset_index(drop=True); df["year"] = df["date"].dt.year
o3 = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")[["date", "o3"]].rename(columns={"o3": "o3_mda8"}).sort_values("date")
o3["date"] = pd.to_datetime(o3["date"])
for k in OZONE_LAGS: o3[f"o3_mda8_lag{k}"] = o3["o3_mda8"].shift(k)
df["date"] = pd.to_datetime(df["date"]); df = df.merge(o3, on="date", how="left")
O3_COLS = ["o3_mda8"] + [f"o3_mda8_lag{k}" for k in OZONE_LAGS]

# the pool, built exactly as in notebook 06 (the PM master's own o3 columns stay excluded)
EXCLUDE = {"date", "pm25", "year", "season"} | set(O3_COLS)
EXCLUDE |= {c for c in df.columns if c == "pm10" or c.startswith("pm10_lag")}
EXCLUDE |= {c for c in df.columns if c == "o3" or c.startswith("o3_lag")}
ALL = [c for c in df.columns if c not in EXCLUDE]
cov = df[ALL].notna().mean(); CANDIDATES = cov[cov >= MIN_COVERAGE].index.tolist()
CAL = {"doy_sin", "doy_cos", "month_sin", "month_cos"}
pool = [c for c in CANDIDATES if c not in CAL | {"pm25_lag1"}]
lead = json.load(open(DATA_DIR / "bogota_feature_selection.json"))["selected_no_lag"]
print(len(pool), "candidates without ozone;", len(pool) + len(O3_COLS), "with it | lead set:", lead)
rows = df.dropna(subset=["o3_mda8"]).reset_index(drop=True)
print("rows: all PM2.5 days", len(df), "-> days with ozone", len(rows))
print("ozone coverage of the added columns on those rows:", rows[O3_COLS].notna().mean().round(3).to_dict())

189 candidates without ozone; 193 with it | lead set: ['w700', 'co', 'hotspots_sector_NE_lag2', 'sensible_heat_flux_lag1', 'wind_speed_era5', 'hotspots_sector_N', 'sensible_heat_flux', 'blh_mean', 'hotspots_sector_E_lag2']
rows: all PM2.5 days 4383 -> days with ozone 3285
ozone coverage of the added columns on those rows: {'o3_mda8': 1.0, 'o3_mda8_lag1': 0.999, 'o3_mda8_lag2': 0.999, 'o3_mda8_lag3': 0.998}


In [3]:
# baseline: the lead PM2.5 feature set, scored on exactly the rows used below
base_rows = loyo_r2(rows, lead, "pm25")
base_all = loyo_r2(df, lead, "pm25")
print(f"lead set, neural net, LOYO pooled R2: all {len(df)} days {base_all:.3f} | the {len(rows)} days with ozone {base_rows:.3f}")

lead set, neural net, LOYO pooled R2: all 4383 days 0.724 | the 3285 days with ozone 0.724


In [4]:
t0 = time.time()
print("=== SFS WITH ozone offered ===")
sel_o3, sc_o3 = sequential_forward_selection(rows, pool + O3_COLS, target="pm25", max_features=MAX_FEATURES, min_gain=MIN_GAIN, n_jobs=N_JOBS)
print("selected:", sel_o3, "\nscores:", [round(s, 3) for s in sc_o3], f"({(time.time()-t0)/60:.1f} min)")
json.dump(dict(selected_with_o3=sel_o3, scores_with_o3=sc_o3, lead_set=lead, lead_rescored_same_rows=base_rows, lead_all_days=base_all, n_rows=len(rows)), open(DATA_DIR / "bogota_feature_selection_with_o3.json", "w"), indent=2)

=== SFS WITH ozone offered ===


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


... (19 outputs left out)

  + tcwv                                -> across-year R2 = 0.727


  + tke_proxy                           -> across-year R2 = 0.732


  + o3_mda8_lag1                        -> across-year R2 = 0.740


selected: ['co', 'o3_mda8', 'hotspots_sector_E_lag2', 'wind_speed_era5', 'day_of_week', 'hotspots_magdalena_valley_lag2', 'tcwv', 'tke_proxy', 'o3_mda8_lag1'] 
scores: [0.362, 0.613, 0.668, 0.702, 0.72, 0.724, 0.727, 0.732, 0.74] (42.1 min)


In [5]:
if RUN_CONTROL:
    t0 = time.time(); print("=== SFS WITHOUT ozone, same rows (control) ===")
    sel_c, sc_c = sequential_forward_selection(rows, pool, target="pm25", max_features=MAX_FEATURES, min_gain=MIN_GAIN, n_jobs=N_JOBS)
    print("selected:", sel_c, "\nscores:", [round(s, 3) for s in sc_c], f"({(time.time()-t0)/60:.1f} min)")
    res = json.load(open(DATA_DIR / "bogota_feature_selection_with_o3.json")); res.update(selected_control=sel_c, scores_control=sc_c)
    json.dump(res, open(DATA_DIR / "bogota_feature_selection_with_o3.json", "w"), indent=2)

=== SFS WITHOUT ozone, same rows (control) ===


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/path/to/project/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


... (19 outputs left out)

  + tke_proxy                           -> across-year R2 = 0.715


  + hotspots_magdalena_valley_lag1      -> across-year R2 = 0.717


  + hotspots_caqueta_lag2               -> across-year R2 = 0.728


selected: ['co', 'hotspots_sector_E_lag2', 'wind_speed_era5', 'sensible_heat_flux', 'hotspots_sector_NE_lag2', 'tke_proxy', 'hotspots_magdalena_valley_lag1', 'hotspots_caqueta_lag2'] 
scores: [0.362, 0.574, 0.647, 0.678, 0.702, 0.715, 0.717, 0.728] (36.4 min)
